# Physarum synaptique — des arêtes passives aux synapses actives

On transforme le réservoir Physarum : les **arêtes passives** (simple loi d'Ohm) deviennent des **synapses actives** qui font du traitement local.

## Le pipeline

```
[ Graphe 14x14 ] ──> 1. Flux Non Linéaire ──> Q_ij = D_ij * tanh(alpha * Δp)
                          │
                          ▼
                   2. Plasticité Hebbienne ──> D_ij += η·(σ(p_i)·σ(p_j)) - γ·D_ij
                          │
                          ▼
                   3. Intégration Dendritique ──> vecteur z (somme ReLU par zone)
                          │
                          ▼
              [ Predictive Coding + Couche Lue ]
```

## Les 3 étapes (plan)
1. **Flux non-linéaire** : `Q_ij = D_ij · tanh(α·(p_i - p_j))` — étouffe le bruit, amplifie les vrais traits (seuil synaptique).
2. **Plasticité Hebbienne locale** (STDP-like) : `D_ij += η·(σ(p_i)·σ(p_j)) - γ·D_ij` — co-activation → le tuyau s'élargit (extraction de corrélations).
3. **Intégration dendritique** (pooling) : `z_k = Σ_{(i,j)∈Zone_k} ReLU(Q_ij)` — vecteur compact z par sous-zones, envoyé au Predictive Coding.

## 0. Imports

In [1]:
# Physarum synaptique — des arêtes passives aux synapses actives
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from recherche_agi import (load_mnist, SynapticReservoir, train_readout,
    synaptic_signature, grid_graph_from_image, PhysarumGraph)
from recherche_agi.synaptic_physarum import (synaptic_flow, hebbian_plasticity,
    dendritic_pooling, _zone_assignment)

## 1. Données : MNIST

In [2]:
train_set, test_set = load_mnist()
print("Train :", len(train_set), "| Test :", len(test_set))

Train : 60000 | Test : 10000


## 2. Étape 1 — Flux non-linéaire (synapse)

In [3]:
# Construire un graphe (grille régulière) pour un chiffre
img, label = test_set[7]
img_np = img.squeeze().numpy()
graph, sources, info = grid_graph_from_image(img_np, downscale=8)
print(f"Graphe {label} : {graph.n_nodes} nœuds, {graph.n_edges} arêtes")
gh, gw = info['gh'], info['gw']
centroids = np.array([[j+0.5, i+0.5] for i in range(gh) for j in range(gw)])

# drains = bord
border = set()
for i in range(gh):
    border.add(i*gw); border.add(i*gw+gw-1)
for j in range(gw):
    border.add(j); border.add((gh-1)*gw+j)
sinks = sorted(border)[:min(10, len(border))]

# Flux linéaire (Ohm) vs flux synaptique (tanh)
p, Q_syn, Q_lin = synaptic_flow(graph, sources, sinks, alpha=5.0)
print(f"\nFlux linéaire max : {abs(Q_lin).max():.4f}")
print(f"Flux synaptique (tanh) max : {abs(Q_syn).max():.4f}")
print("\nLe tanh étouffe les petites différences de pression (bruit) et sature")
print("les fortes différences (les vrais traits du chiffre).")

Graphe 9 : 81 nœuds, 144 arêtes

Flux linéaire max : 0.1408
Flux synaptique (tanh) max : 0.4435

Le tanh étouffe les petites différences de pression (bruit) et sature
les fortes différences (les vrais traits du chiffre).


## 3. Étape 2 — Plasticité Hebbienne locale

In [4]:
# Itérer flux synaptique + plasticité Hebbienne (le réseau "apprend" localement)
print("=== Évolution des conductances (plasticité Hebbienne) ===")
graph2, sources2, info2 = grid_graph_from_image(img_np, downscale=8)
for it in range(6):
    p, Q, _ = synaptic_flow(graph2, sources2, sinks, alpha=5.0)
    hebbian_plasticity(graph2, p, eta=0.1, gamma=0.1, beta=5.0)
    print(f"  iter {it+1} : conductances min={min(graph2.D):.4f} max={max(graph2.D):.4f} "
          f"moy={np.mean(graph2.D):.3f}")
print("\nLes arêtes dont les nœuds sont co-activés (sous pression ensemble) se sont")
print("élargies : extraction locale des corrélations de forme.")

=== Évolution des conductances (plasticité Hebbienne) ===
  iter 1 : conductances min=0.4750 max=0.5486 moy=0.532
  iter 2 : conductances min=0.4525 max=0.5919 moy=0.560
  iter 3 : conductances min=0.4323 max=0.6306 moy=0.585
  iter 4 : conductances min=0.4140 max=0.6651 moy=0.608
  iter 5 : conductances min=0.3976 max=0.6960 moy=0.628
  iter 6 : conductances min=0.3829 max=0.7234 moy=0.646

Les arêtes dont les nœuds sont co-activés (sous pression ensemble) se sont
élargies : extraction locale des corrélations de forme.


## 4. Étape 3 — Intégration dendritique (pooling par zones)

In [5]:
# Regrouper les flux par sous-zones spatiales -> vecteur z compact
z = dendritic_pooling(graph2, Q, centroids, n_zones=64)
print(f"Vecteur z (pooling 64 zones) : {z.shape}")
print(f"z normalisé : {np.round(z/np.linalg.norm(z), 3)}")

# Visualiser les zones
zones = _zone_assignment(centroids, 64)
print(f"\nZones spatiales (64) : chaque nœud est assigné à une zone de la grille 8x8")

Vecteur z (pooling 64 zones) : (64,)
z normalisé : [0.    0.307 0.044 0.    0.    0.    0.    0.    0.    0.925 0.196 0.
 0.    0.    0.    0.    0.    0.023 0.094 0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.   ]

Zones spatiales (64) : chaque nœud est assigné à une zone de la grille 8x8


## 5. Réservoir synaptique — signature z + couche lue

Le réservoir synaptique combine les 3 étapes et produit une **signature compacte** `z` par image, qui alimente une couche de lecture entraînée (et le Predictive Coding).

In [6]:
# Réservoir synaptique (64 zones, meilleur réglage)
reservoir = SynapticReservoir(alpha=5.0, n_iter=10, n_zones=64, downscale=8,
                              eta=0.1, gamma=0.1, beta=5.0)

def extract(dataset, n):
    X, y = [], []
    cnt = [0]*10
    for i in range(len(dataset)):
        l = int(dataset[i][1])
        if cnt[l] >= n//10: continue
        X.append(reservoir.signature(dataset[i][0].squeeze().numpy()))
        y.append(l); cnt[l] += 1
        if sum(cnt) >= n: break
    return np.array(X), np.array(y)

Xtr, ytr = extract(train_set, 300)
Xte, yte = extract(test_set, 150)
print(f"Signatures z : train {Xtr.shape}, test {Xte.shape}")

Signatures z : train (300, 64), test (150, 64)


In [7]:
# Couche lue sur le vecteur z
readout = train_readout(Xtr, ytr, n_classes=10, epochs=80)
with torch.no_grad():
    acc_syn = (readout(torch.tensor(Xte, dtype=torch.float32)).argmax(1) == torch.tensor(yte)).float().mean().item()
print(f"Précision couche lue sur z synaptique : {acc_syn:.3f}")

Précision couche lue sur z synaptique : 0.393


## 6. Comparaison avec les représentations précédentes

| Représentation | Signature | Acc couche lue |
|---|---|---|
| Grille régulière (flux linéaire) | 24 dims | 0.45 |
| Superpixels | 50 dims | 0.245 |
| **Physarum synaptique** | **64 dims** | **~0.40** |

In [8]:
print("=== SYNTHÈSE ===")
print(f"Réservoir synaptique (tanh + Hebbien + pooling) : acc {acc_syn:.3f}")
print("Le vecteur z compact (64 dims) est comparable à la grille régulière brute")
print("mais avec des arêtes ACTIVES (synapses) qui font du traitement local :")
print("  - tanh : seuil de déclenchement (étouffe le bruit)")
print("  - Hebbien : co-activation renforce les tuyaux (corrélations de forme)")
print("  - pooling : intégration dendritique (somme ReLU par zone)")
print("Cette signature compacte est prête pour le Predictive Coding.")

=== SYNTHÈSE ===
Réservoir synaptique (tanh + Hebbien + pooling) : acc 0.393
Le vecteur z compact (64 dims) est comparable à la grille régulière brute
mais avec des arêtes ACTIVES (synapses) qui font du traitement local :
  - tanh : seuil de déclenchement (étouffe le bruit)
  - Hebbien : co-activation renforce les tuyaux (corrélations de forme)
  - pooling : intégration dendritique (somme ReLU par zone)
Cette signature compacte est prête pour le Predictive Coding.
